In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

In [0]:
  #path = "`medallion-data-pipeline_data`.bronze.dirty_data_s3"

In [0]:
@dp.materialized_view(
  name = "`medallion-data-pipeline_data`.silver.cleaned_data_s3",
  comment = "Clean user data for silver layer"
)

def cleaned_data_s3():
  df = spark.read.table("`medallion-data-pipeline_data`.bronze.dirty_data_s3")
  
  ## Drop column _rescued_data
  df = df.drop('_rescued_data')

  ## Convert signup_date to datetime
  df = df.withColumn("signup_date", 
      F.to_date(F.col("signup_date"), "MM/dd/yy"))

  ## Drop duplicates based on user_id, keeping the first occurences 
  df = df.dropDuplicates(["user_id"])

  return df

In [0]:
# df_clean_spark = spark.createDataFrame(df)

# df_clean_spark.write.mode("overwrite").saveAsTable("`medallion-data-pipeline_data`.silver.cleaned_data_s3")

In [0]:
@dp.materialized_view(
    name = "`medallion-data-pipeline_data`.gold.signups_by_day",
    comment = "Insights on sign ups by day"
)
def signups_by_day():
    df = spark.read.table("`medallion-data-pipeline_data`.silver.cleaned_data_s3")

    df = df.withColumn('week_day', F.date_format(F.col('signup_date'), 'EEEE'))

    df = df.groupBy('week_day', 'referral_source')\
        .agg(
            F.count('transaction_id').alias('signups')
        )

    return df